# 01 — Data Loading and Epoching

This notebook loads Sleep-EDF hypnogram annotations and converts them into a clean 30-second epoch-level sleep-stage table.

## Objectives

- find local Sleep-EDF hypnogram files;
- inspect available PSG/Hypnogram file pairs;
- map original annotation labels to standard sleep-stage labels;
- expand annotation intervals into 30-second epochs;
- save the epoch-level dataset for downstream sleep-staging models.


In [ ]:
from pathlib import Path
import re

import pandas as pd
import matplotlib.pyplot as plt
import mne
from IPython.display import display


In [ ]:
from pathlib import Path


def find_project_root(start: Path = Path.cwd()) -> Path:
    """Find project root by walking up to README.md and data/."""
    for path in [start, *start.parents]:
        if (path / 'README.md').exists() and (path / 'data').exists():
            return path
    raise FileNotFoundError(
        'Project root not found. Run this notebook from inside the project folder.'
    )


PROJECT_ROOT = find_project_root()
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
FIGURES_DIR = PROJECT_ROOT / 'figures'
MODELS_DIR = PROJECT_ROOT / 'models'
NOTEBOOKS_DIR = PROJECT_ROOT / 'notebooks'

SLEEP_EDF_DIR = RAW_DIR / 'sleep_edf' / 'sleep-cassette'

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT =', PROJECT_ROOT)
print('SLEEP_EDF_DIR =', SLEEP_EDF_DIR)
print('SLEEP_EDF_DIR exists =', SLEEP_EDF_DIR.exists())


## Locate Sleep-EDF files

The mini-version of the project expects local Sleep-EDF files. Hypnogram files are required for this notebook; PSG files are listed only to check available pairs.


In [ ]:
edf_files = sorted(SLEEP_EDF_DIR.glob("*.edf")) + sorted(SLEEP_EDF_DIR.glob("*.EDF"))

if not edf_files:
    raise FileNotFoundError(f"No EDF files found in {SLEEP_EDF_DIR.relative_to(PROJECT_ROOT)}")

file_inventory = pd.DataFrame({
    "file_name": [path.name for path in edf_files],
    "file_type": [
        "PSG" if "-PSG" in path.name else "Hypnogram" if "Hypnogram" in path.name else "Other"
        for path in edf_files
    ],
})

display(file_inventory)


In [ ]:
def get_subject_prefix(filename: str) -> str | None:
    match = re.match(r"^(SC\d{4})", filename)
    return match.group(1) if match else None


psg_files = {}
hypnogram_files = {}

for file_path in edf_files:
    prefix = get_subject_prefix(file_path.name)

    if prefix is None:
        continue

    if "-PSG.edf" in file_path.name or "-PSG.EDF" in file_path.name:
        psg_files[prefix] = file_path
    elif "Hypnogram" in file_path.name:
        hypnogram_files[prefix] = file_path

pairs = []

for subject_id in sorted(hypnogram_files):
    pairs.append({
        "subject_id": subject_id,
        "psg_file": psg_files.get(subject_id),
        "hypnogram_file": hypnogram_files[subject_id],
        "has_psg": subject_id in psg_files,
    })

pairs_df = pd.DataFrame([
    {
        "subject_id": row["subject_id"],
        "psg_file": row["psg_file"].name if row["psg_file"] is not None else None,
        "hypnogram_file": row["hypnogram_file"].name,
        "has_psg": row["has_psg"],
    }
    for row in pairs
])

display(pairs_df)


## Load and map hypnogram annotations

Sleep-EDF annotations are mapped to compact stage labels:

- `W` — wake;
- `N1`, `N2`, `N3` — non-REM stages;
- `REM` — REM sleep.

Movement and unknown annotations are excluded from the modeling table.


In [ ]:
STAGE_MAP = {
    "Sleep stage W": "W",
    "Sleep stage 1": "N1",
    "Sleep stage 2": "N2",
    "Sleep stage 3": "N3",
    "Sleep stage 4": "N3",
    "Sleep stage R": "REM",
}


def annotations_to_epochs(
    hypnogram_path: Path,
    subject_id: str,
    epoch_duration_sec: int = 30,
) -> pd.DataFrame:
    """Convert one Sleep-EDF hypnogram file into a 30-second epoch table."""
    annotations = mne.read_annotations(hypnogram_path)

    rows = []

    for onset, duration, description in zip(
        annotations.onset,
        annotations.duration,
        annotations.description,
    ):
        stage_label = STAGE_MAP.get(description)

        if stage_label is None:
            continue

        n_epochs = int(duration // epoch_duration_sec)

        for epoch_idx in range(n_epochs):
            rows.append({
                "subject_id": subject_id,
                "epoch_start_sec": float(onset + epoch_idx * epoch_duration_sec),
                "epoch_duration_sec": epoch_duration_sec,
                "stage_label": stage_label,
                "original_description": description,
            })

    return pd.DataFrame(rows)


epoch_tables = []

for pair in pairs:
    epoch_table = annotations_to_epochs(
        hypnogram_path=pair["hypnogram_file"],
        subject_id=pair["subject_id"],
    )
    epoch_tables.append(epoch_table)

epoch_df = pd.concat(epoch_tables, ignore_index=True)

display(epoch_df.head())
display(epoch_df.shape)


## Stage distribution

The stage distribution is inspected before downstream modeling. Strong class imbalance is expected in sleep-staging tasks, especially for N1.


In [ ]:
stage_counts = (
    epoch_df["stage_label"]
    .value_counts()
    .rename_axis("stage_label")
    .reset_index(name="n_epochs")
)

stage_counts["percent"] = stage_counts["n_epochs"] / stage_counts["n_epochs"].sum() * 100

display(stage_counts)

subject_counts = (
    epoch_df["subject_id"]
    .value_counts()
    .rename_axis("subject_id")
    .reset_index(name="n_epochs")
)

display(subject_counts)


In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(stage_counts["stage_label"], stage_counts["n_epochs"])
plt.xlabel("Sleep stage")
plt.ylabel("Number of 30-second epochs")
plt.title("Sleep-EDF Epoch Stage Distribution")
plt.tight_layout()

stage_plot_path = FIGURES_DIR / "sleep_edf_epoch_stage_distribution.png"
plt.savefig(stage_plot_path, dpi=150)

plt.show()


## Example hypnogram

A simple hypnogram is plotted for the first available subject. This gives a quick visual check of the temporal sleep-stage structure.


In [ ]:
STAGE_ORDER = {
    "W": 4,
    "REM": 3,
    "N1": 2,
    "N2": 1,
    "N3": 0,
}

example_subject = epoch_df["subject_id"].iloc[0]
example_df = epoch_df[epoch_df["subject_id"] == example_subject].copy()
example_df["stage_numeric"] = example_df["stage_label"].map(STAGE_ORDER)

plt.figure(figsize=(12, 4))
plt.step(
    example_df["epoch_start_sec"] / 3600,
    example_df["stage_numeric"],
    where="post",
)
plt.yticks(
    list(STAGE_ORDER.values()),
    list(STAGE_ORDER.keys()),
)
plt.xlabel("Time from recording start, hours")
plt.ylabel("Sleep stage")
plt.title(f"Example Hypnogram: {example_subject}")
plt.tight_layout()

hypnogram_path = FIGURES_DIR / f"{example_subject}_hypnogram.png"
plt.savefig(hypnogram_path, dpi=150)

plt.show()


## Save epoch index

The epoch-level table is saved for the next notebook, where EEG spectral features are extracted and used for baseline sleep-stage classification.


In [ ]:
output_path = PROCESSED_DIR / "sleep_edf_epoch_index.csv"
epoch_df.to_csv(output_path, index=False)

print(f"Saved epoch table to: {output_path.relative_to(PROJECT_ROOT)}")
print(f"Number of subjects: {epoch_df['subject_id'].nunique()}")
print(f"Number of epochs: {len(epoch_df)}")


## Summary

This notebook converts Sleep-EDF hypnogram annotations into a clean 30-second epoch-level dataset. The resulting file, `data/processed/sleep_edf_epoch_index.csv`, is used by the baseline sleep-staging notebook.

The dataset remains a mini-version intended for pipeline validation and portfolio demonstration.
